# Notebook 02 — Preprocessing NLP avec PySpark

**Phase 1 — Étape 2 : Nettoyage et vectorisation du dataset Sentiment140**

## 📌 Objectif de ce notebook

Ce notebook transforme les tweets bruts du dataset Sentiment140 en données exploitables par le modèle ML :

```
CSV brut (1,6M tweets)
    ↓  Chargement PySpark
    ↓  Nettoyage textuel (URLs, @mentions, ponctuation)
    ↓  Encodage variable cible (0/4 → 0/1)
    ↓  Tokenisation
    ↓  Suppression stopwords
    ↓  TF-IDF (HashingTF + IDF)
    ↓  Sauvegarde Parquet
```

**À exécuter APRÈS :** `01_exploration_sentiment140.ipynb`  
**À exécuter AVANT :** `03_training_model_spark_mllib.ipynb`

## 🐳 Mode d'exécution — Docker

Ce notebook s'exécute dans le conteneur `spark-submit` (plus de Spark local Windows).  
Lancer avec la commande suivante dans PowerShell :

```powershell
docker exec spark-submit jupyter nbconvert --to notebook --execute notebooks/02_preprocessing_sentiment140_pyspark.ipynb --output notebooks/02_output.ipynb
```

Ou pour l'exécution interactive depuis Jupyter dans Docker :

```powershell
docker exec -it spark-submit jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root
```
Puis ouvrir http://localhost:8888 dans votre navigateur.

> 📢 **Note soutenance** : Section 4.2 du rapport — Pipeline NLP batch + encodage variable cible

## 0. Vérification de l'environnement Docker

Avant d'exécuter ce notebook, vérifiez que :
- ✅ `docker-compose up -d` a été lancé
- ✅ `docker-compose --profile tools up -d spark-submit` a été lancé
- ✅ `data/raw/sentiment140.csv` existe dans le dossier projet Windows
- ✅ Le notebook 01 a été exécuté (exploration OK)

> ❌ **Plus besoin de** : Java local, HADOOP_HOME, winutils.exe, findspark

In [1]:
# ── Vérification de l'environnement ──────────────────────────
import sys
import os
import subprocess
from pathlib import Path

print('🐍 Python version :', sys.version)

# Détection Docker
in_docker = os.path.exists('/.dockerenv')
print(f'🐳 Environnement Docker : {in_docker}')

if not in_docker:
    print('💻 Mode Windows local détecté')
    print('   → PySpark doit être installé dans le venv Windows')
    print('   → Commande : pip install pyspark==3.5.1 pyarrow')

# Vérification PySpark
try:
    import pyspark
    print('⚡ PySpark version :', pyspark.__version__)
except ImportError:
    print('❌ PySpark manquant !')
    print('   Exécutez dans PowerShell (venv activé) :')
    print('   pip install pyspark==3.5.1 pyarrow findspark')
    raise SystemExit('Installez PySpark avant de continuer')

# ROOT_DIR
ROOT_DIR = Path('/workspace') if in_docker else Path().absolute().parent
print(f'📂 ROOT_DIR : {ROOT_DIR}')

# Vérification CSV
csv_path = ROOT_DIR / 'data' / 'raw' / 'sentiment140.csv'
if csv_path.exists():
    size_mb = csv_path.stat().st_size / 1024 / 1024
    print(f'✅ sentiment140.csv trouvé ({size_mb:.1f} MB)')
else:
    print(f'❌ Fichier manquant : {csv_path}')
    raise SystemExit('Placez sentiment140.csv dans data/raw/')


🐍 Python version : 3.8.10 (default, Nov 22 2023, 10:22:35) 
[GCC 9.4.0]
🐳 Environnement Docker : True
❌ PySpark manquant !
   Exécutez dans PowerShell (venv activé) :
   pip install pyspark==3.5.1 pyarrow findspark


AttributeError: 'tuple' object has no attribute 'tb_frame'

## 12. Synthèse du Notebook 02

| Étape | Action | Résultat |
|-------|--------|----------|
| Chargement | CSV latin-1 → DataFrame Spark | 1 600 000 tweets |
| Sélection | 6 colonnes → 2 colonnes (text, label) | Simplification ML |
| Encodage | target {0,4} → label {0,1} | Format MLlib |
| Nettoyage | URLs, @, #, ponctuation supprimés | Bruit éliminé |
| Sauvegarde | Parquet Snappy | ~50-80 MB (compressé) |

### ✅ Compatibilité Windows local ET Docker

| Élément | Windows local | Docker |
|---------|--------------|--------|
| PySpark | `pip install pyspark` dans venv | Intégré dans l'image |
| findspark | Utilisé si disponible (optionnel) | Non utilisé |
| SPARK_MASTER | `local[*]` (config.py) | `spark://spark-master:7077` |
| Parquet | Peut échouer (winutils) | Fonctionne toujours ✅ |
| matplotlib | Mode interactif | Mode `Agg` (sans écran) |

**Points importants pour la soutenance :**
- Le TF-IDF n'est pas sauvegardé séparément — il sera **intégré dans le pipeline ML complet** (Notebook 03)
- Le Parquet nettoyé sera rechargé par le Notebook 03 sans recalcul
- Le preprocessing est **identique** pour Sentiment140 et Apple Tweets (même UDF `clean_text`)
- La colonne Apple `tweet_text` sera simplement renommée `text` — aucun autre changement


In [ ]:
import sys
import re
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Détection Docker vs Windows ────────────────────────────────
in_docker = os.path.exists('/.dockerenv')
ROOT_DIR = Path('/workspace') if in_docker else Path().absolute().parent
sys.path.insert(0, str(ROOT_DIR))

# ── Compatibilité Windows : findspark si PySpark non trouvé ────
# Sur Windows local, PySpark peut nécessiter findspark
# Dans Docker, findspark n'est pas nécessaire (PySpark natif)
if not in_docker:
    try:
        import findspark
        findspark.init()
        print('✅ findspark initialisé (mode Windows local)')
    except ImportError:
        pass  # findspark optionnel si PySpark directement dans le venv

# ── Imports PySpark ────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType
from pyspark.ml.feature import (
    Tokenizer,
    StopWordsRemover,
    HashingTF,
    IDF
)
from pyspark.ml import Pipeline

# ── Imports projet ─────────────────────────────────────────────
from config.config import (
    SENTIMENT140_CSV,
    SENTIMENT140_CLEAN_PARQUET,
    S140_COLUMNS,
    TFIDF_NUM_FEATURES,
    SPARK_MASTER,
    SPARK_DRIVER_MEMORY,
    SPARK_EXECUTOR_MEMORY
)
from src.utils import get_logger, ensure_dirs, clean_text_udf

logger = get_logger('preprocessing_notebook')
print('✅ Tous les imports réussis')
print(f'   Mode          : {"Docker" if in_docker else "Windows local"}')
print(f'   SPARK_MASTER  : {SPARK_MASTER}')
print(f'   CSV           : {SENTIMENT140_CSV}')
print(f'   Parquet output: {SENTIMENT140_CLEAN_PARQUET}')


## 2. Création de la SparkSession

La SparkSession est le point d'entrée unique de tout traitement Spark.

### ✅ Changement Docker

```python
# AVANT (Windows local — hardcodé)
.master('local[*]')

# APRÈS (Docker — lu depuis config.py via env var)
.master(SPARK_MASTER)  # = 'spark://spark-master:7077' dans Docker
```

> Dans Docker, `SPARK_MASTER` est injecté automatiquement par `docker-compose.yml`  
> Le Spark Master UI est visible sur http://localhost:9090

In [ ]:
# ── Configuration SparkSession — Windows local ET Docker ───────
import os

in_docker = os.path.exists('/.dockerenv')

# Sur Windows local : SPARK_MASTER = 'local[*]' (depuis config.py)
# Dans Docker      : SPARK_MASTER = 'spark://spark-master:7077'
builder = (
    SparkSession.builder
    .master(SPARK_MASTER)
    .appName('SentimentAnalysis_Preprocessing')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', SPARK_DRIVER_MEMORY)
    .config('spark.executor.memory', SPARK_EXECUTOR_MEMORY)
    .config('spark.driver.maxResultSize', '2g')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .config('spark.ui.showConsoleProgress', 'false')
)

# Sur Windows : configure les chemins temporaires pour éviter les
# erreurs winutils (Spark écrit dans le dossier temporaire Python)
if not in_docker:
    import tempfile
    tmp = tempfile.gettempdir().replace('\\', '/')
    builder = builder \
        .config('spark.local.dir', tmp) \
        .config('spark.sql.warehouse.dir', tmp + '/spark-warehouse')

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')

print('✅ SparkSession créée')
print(f'   Version Spark : {spark.version}')
print(f'   Master        : {spark.sparkContext.master}')
print(f'   App Name      : {spark.sparkContext.appName}')
if in_docker:
    print('   Spark UI      : http://localhost:9090 (Docker Master)')
else:
    print('   Spark UI      : http://localhost:4040 (Windows local)')


## 3. Chargement du dataset Sentiment140 avec PySpark

### Pourquoi PySpark plutôt que Pandas ?

| Critère | Pandas | PySpark |
|---------|--------|---------|
| Volume max | ~1-2 GB (RAM) | Illimité (distribué) |
| 1,6M tweets | Lent / OOM | Rapide ✅ |
| Intégration Kafka | Non | Oui ✅ |
| Entraînement ML | sklearn | MLlib distribué ✅ |

Sentiment140 fait ~230 MB — **PySpark est obligatoire** pour être cohérent avec le pipeline Streaming qui suit.

### 🐳 Chemin Docker
Le fichier CSV est dans `/workspace/data/raw/sentiment140.csv` dans le conteneur  
(monté depuis `./data/raw/sentiment140.csv` sur Windows via volume Docker)

In [ ]:
# ── Chargement CSV → DataFrame Spark ───────────────────────────
print('⏳ Chargement de Sentiment140 (peut prendre 30-60s)...')
print(f'   Chemin : {SENTIMENT140_CSV}')

df_raw = (
    spark.read
    .option('header', 'false')      # Pas d'en-tête dans Sentiment140
    .option('encoding', 'latin-1')  # Encodage spécifique Sentiment140
    .option('multiLine', 'false')   # Un tweet = une ligne
    .option('quote', '"')
    .option('escape', '"')
    .csv(str(SENTIMENT140_CSV))
    .toDF(*S140_COLUMNS)            # Nommage des 6 colonnes
)

total = df_raw.count()
print(f'✅ Dataset chargé : {total:,} tweets × {len(df_raw.columns)} colonnes')
print()
print('Aperçu du DataFrame Spark :')
df_raw.show(5, truncate=70)

In [ ]:
# ── Schéma des colonnes ─────────────────────────────────────────
print('Schéma du DataFrame :')
df_raw.printSchema()

print('\nDistribution de la variable cible (target) :')
df_raw.groupBy('target').count().orderBy('target').show()

## 4. Sélection des colonnes utiles pour le ML

Le modèle ML utilise **uniquement 2 colonnes** sur les 6 disponibles :
- `text` → feature d'entrée (transformée en TF-IDF)
- `target` → variable cible (recodée en label binaire)

Les colonnes `id`, `date`, `flag`, `user` sont écartées intentionnellement.

In [ ]:
# ── Sélection text + target uniquement ─────────────────────────
df_selected = df_raw.select('text', 'target')

print('DataFrame après sélection des colonnes ML :')
df_selected.show(3, truncate=80)

print(f'Colonnes utilisées : {df_selected.columns}')
print(f'Colonnes écartées  : {[c for c in S140_COLUMNS if c not in ["text", "target"]]}')

## 5. Encodage de la variable cible

Sentiment140 utilise une convention non-standard :
- `target = 0` → tweet négatif
- `target = 4` → tweet positif (pas 1 !)

La Logistic Regression de Spark MLlib attend un label binaire **{0, 1}**.

```
target = 0  →  label = 0  (Négatif)
target = 4  →  label = 1  (Positif)
```

In [ ]:
# ── Encodage : target {0, 4} → label {0, 1} ────────────────────
df_labeled = df_selected.withColumn(
    'label',
    F.when(F.col('target') == '4', 1)
     .otherwise(0)
     .cast(IntegerType())
).drop('target')  # Suppression de l'ancienne colonne target

print('Distribution des labels après encodage :')
df_labeled.groupBy('label').count().show()

# Vérification
n_pos = df_labeled.filter(F.col('label') == 1).count()
n_neg = df_labeled.filter(F.col('label') == 0).count()
print(f'✅ label=0 (Négatif) : {n_neg:,} tweets ({n_neg/total*100:.1f}%)')
print(f'✅ label=1 (Positif) : {n_pos:,} tweets ({n_pos/total*100:.1f}%)')
print(f'   Dataset parfaitement équilibré : pas de rééquilibrage nécessaire')

## 6. Nettoyage textuel

### Étapes du pipeline NLP

| Étape | Transformation | Exemple |
|-------|---------------|--------|
| 1 | Lowercasing | "Apple" → "apple" |
| 2 | Suppression URLs | "http://t.co/abc" → "" |
| 3 | Suppression @mentions | "@Apple" → "" |
| 4 | Suppression # symbole | "#iPhone" → "iPhone" |
| 5 | Suppression ponctuation | "great!" → "great" |
| 6 | Normalisation espaces | "  " → " " |

### ✅ Compatibilité Docker
La fonction `clean_text` est dans `src/utils.py`, monté en volume `/workspace/src/utils.py`.  
L'UDF Spark (`clean_text_udf`) est enregistrée depuis ce module — **aucun changement de code nécessaire**.

In [ ]:
# ── Démonstration du nettoyage sur des exemples ─────────────────
from src.utils import clean_text

exemples = [
    "@Apple I LOVE my new iPhone!! Check http://apple.com #iPhone",
    "This is absolutely terrible... worst experience EVER!!!",
    "Just got home from work. #tired @friend hello world",
    "The new iOS update is amazing!! Really happy :)"
]

print('=== DÉMONSTRATION DU NETTOYAGE TEXTUEL ===')
print(f'{"Tweet brut":<55} → {"Après nettoyage"}')
print('-' * 90)
for tweet in exemples:
    cleaned = clean_text(tweet)
    print(f'{tweet[:53]:<55} → {cleaned}')

In [ ]:
# ── Application du nettoyage sur tout le dataset avec UDF Spark ─
print('⏳ Nettoyage textuel sur 1,6M tweets...')

df_cleaned = (
    df_labeled
    .withColumn('text_clean', clean_text_udf(F.col('text')))
    .filter(F.length(F.col('text_clean')) > 3)   # Suppression tweets vides après nettoyage
    .drop('text')
    .withColumnRenamed('text_clean', 'text')
)

n_cleaned = df_cleaned.count()
n_removed = total - n_cleaned
print(f'✅ Nettoyage terminé')
print(f'   Tweets conservés : {n_cleaned:,}')
print(f'   Tweets supprimés (trop courts après nettoyage) : {n_removed:,}')

print('\nAperçu après nettoyage :')
df_cleaned.show(5, truncate=80)

## 7. Pipeline TF-IDF

### Formules mathématiques

**Term Frequency (TF)** : fréquence relative d'un terme dans un document
$$TF(t, d) = \frac{\text{occurrences de } t \text{ dans } d}{\text{nombre total de termes dans } d}$$

**Inverse Document Frequency (IDF)** : pénalise les termes trop fréquents
$$IDF(t, D) = \log\left(\frac{|D|+1}{|\{d \in D : t \in d\}|+1}\right) + 1$$

**TF-IDF** : produit des deux → termes importants et distinctifs
$$TFIDF(t, d, D) = TF(t, d) \times IDF(t, D)$$

### Pourquoi HashingTF avec 2^18 features ?
- `2^18 = 262 144` dimensions → couvre le vocabulaire Twitter complet
- HashingTF est **déterministe** → même hash = même feature à l'inférence
- Pas besoin de stocker le vocabulaire → compatible streaming

In [ ]:
# ── Construction du pipeline NLP complet ───────────────────────
print(f'Construction du pipeline TF-IDF (numFeatures={TFIDF_NUM_FEATURES:,})...')

# Étape 1 : Tokenisation (découpage en mots)
tokenizer = Tokenizer(
    inputCol='text',
    outputCol='tokens_raw'
)

# Étape 2 : Suppression des stopwords ("the", "is", "a", ...)
sw_remover = StopWordsRemover(
    inputCol='tokens_raw',
    outputCol='tokens',
    caseSensitive=False
)

# Étape 3 : HashingTF — conversion tokens → vecteur de fréquences
hashing_tf = HashingTF(
    inputCol='tokens',
    outputCol='raw_features',
    numFeatures=TFIDF_NUM_FEATURES   # 2^18 = 262 144
)

# Étape 4 : IDF — pondération par fréquence inverse
idf = IDF(
    inputCol='raw_features',
    outputCol='features',
    minDocFreq=2   # Ignore les termes apparus dans moins de 2 documents
)

# Assemblage du pipeline NLP
nlp_pipeline = Pipeline(stages=[tokenizer, sw_remover, hashing_tf, idf])

print('✅ Pipeline NLP construit')
for i, stage in enumerate(nlp_pipeline.getStages()):
    print(f'   Stage {i+1}: {stage.__class__.__name__}')

In [ ]:
# ── Entraînement du pipeline NLP (fit sur les données) ─────────
print('⏳ Ajustement du pipeline TF-IDF sur 1,6M tweets...')
print('   (Calcul des IDF sur tout le corpus — peut prendre 2-5 minutes)')

nlp_model = nlp_pipeline.fit(df_cleaned)

print('✅ Pipeline TF-IDF ajusté')

In [ ]:
# ── Transformation : texte → vecteurs TF-IDF ───────────────────
print('⏳ Transformation des tweets en vecteurs TF-IDF...')

df_features = nlp_model.transform(df_cleaned)

print('Colonnes après transformation :')
for col in df_features.columns:
    print(f'  - {col}')

# Aperçu d'un vecteur TF-IDF
print('\nExemple de vecteur TF-IDF (sparse) :')
sample = df_features.select('text', 'tokens', 'features').limit(2).collect()
for row in sample:
    print(f'\nTweet : {row["text"][:60]}')
    print(f'Tokens: {row["tokens"][:8]}')
    print(f'TF-IDF: {str(row["features"])[:100]}...')

## 8. Sauvegarde en Parquet

Le format **Parquet** est choisi pour :
- Stockage colonnaire → lecture partielle efficace
- Compression Snappy → 3-10x moins d'espace que CSV
- Schéma intégré → types garantis à la relecture
- Natif Spark → lecture/écriture ultra-rapide

**Important** : On sauvegarde le DataFrame **nettoyé** (text + label), pas les features TF-IDF.  
Le pipeline TF-IDF sera intégré dans le modèle ML complet (Notebook 03).

### 🐳 Écriture Parquet dans Docker

Spark écrit dans `/workspace/data/processed/` à l'intérieur du conteneur.  
Ce dossier est **synchronisé automatiquement** avec `data\processed\` sur Windows (volume monté).  
**Aucune erreur winutils / HADOOP_HOME** car Spark tourne dans Linux.

In [ ]:
# ── Sauvegarde du DataFrame nettoyé en Parquet ──────────────────
ensure_dirs(SENTIMENT140_CLEAN_PARQUET.parent)

# On sauvegarde uniquement text + label (pas les vecteurs TF-IDF)
df_to_save = df_cleaned.select('text', 'label')

print(f'⏳ Sauvegarde en Parquet → {SENTIMENT140_CLEAN_PARQUET}')
print(f'   Compression : Snappy')
print(f'   Chemin Docker : /workspace/data/processed/sentiment140_clean.parquet')
print(f'   Chemin Windows : .\\data\\processed\\sentiment140_clean.parquet')

df_to_save.write \
    .mode('overwrite') \
    .parquet(str(SENTIMENT140_CLEAN_PARQUET))

print('✅ Parquet sauvegardé avec succès')

# Vérification de la taille
import os
parquet_files = list(SENTIMENT140_CLEAN_PARQUET.glob('*.parquet'))
total_size = sum(f.stat().st_size for f in parquet_files)
print(f'   Fichiers générés : {len(parquet_files)}')
print(f'   Taille totale    : {total_size/1024/1024:.1f} MB')

## 9. Vérification de la sauvegarde

In [ ]:
# ── Rechargement du Parquet pour vérification ───────────────────
print('Vérification du Parquet sauvegardé...')

df_verify = spark.read.parquet(str(SENTIMENT140_CLEAN_PARQUET))

print(f'✅ Parquet rechargeable correctement')
print(f'   Nombre de lignes : {df_verify.count():,}')
print(f'   Colonnes         : {df_verify.columns}')
print()

print('Schéma Parquet :')
df_verify.printSchema()

print('Distribution des labels dans le Parquet :')
df_verify.groupBy('label').count().orderBy('label').show()

print('Aperçu des 5 premières lignes :')
df_verify.show(5, truncate=80)

## 10. Analyse exploratoire post-nettoyage

In [ ]:
# ── Statistiques sur la longueur des tweets nettoyés ────────────
import os
import pandas as pd
import matplotlib
# Backend non-interactif dans Docker (pas d'écran), interactif sinon
if os.path.exists('/.dockerenv'):
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

df_stats = df_cleaned.withColumn('text_len', F.length(F.col('text')))

stats = df_stats.select('text_len').describe().toPandas()
print('=== STATISTIQUES LONGUEUR DES TWEETS NETTOYÉS ===')
print(stats.to_string())


In [ ]:
# ── Top mots les plus fréquents par classe ──────────────────────
from pyspark.sql.functions import explode, split

print('=== TOP 15 MOTS — TWEETS NÉGATIFS ===')
(
    df_cleaned
    .filter(F.col('label') == 0)
    .select(explode(split(F.col('text'), ' ')).alias('word'))
    .filter(F.length(F.col('word')) > 3)
    .groupBy('word').count()
    .orderBy(F.desc('count'))
    .limit(15)
    .show()
)

print('=== TOP 15 MOTS — TWEETS POSITIFS ===')
(
    df_cleaned
    .filter(F.col('label') == 1)
    .select(explode(split(F.col('text'), ' ')).alias('word'))
    .filter(F.length(F.col('word')) > 3)
    .groupBy('word').count()
    .orderBy(F.desc('count'))
    .limit(15)
    .show()
)

## 11. Fermeture de la SparkSession

In [ ]:
# ── Fermeture propre ────────────────────────────────────────────
import os
in_docker = os.path.exists('/.dockerenv')

spark.stop()
print('✅ SparkSession fermée proprement')
print()
print('═' * 60)
print('RÉCAPITULATIF — Notebook 02 terminé avec succès')
print('═' * 60)
print(f'✅ Dataset chargé    : {total:,} tweets')
print(f'✅ Après nettoyage   : {n_cleaned:,} tweets conservés')
print(f'✅ Parquet sauvegardé: {SENTIMENT140_CLEAN_PARQUET}')
print(f'✅ Colonnes output   : text (str) + label (int 0/1)')
print()
print('📌 Prochaine étape : Notebook 03 — Entraînement MLlib')
if in_docker:
    print('   docker exec spark-submit python src/train_model.py')
else:
    print('   Exécutez le notebook 03 depuis Jupyter')


## 12. Synthèse du Notebook 02

| Étape | Action | Résultat |
|-------|--------|----------|
| Chargement | CSV latin-1 → DataFrame Spark | 1 600 000 tweets |
| Sélection | 6 colonnes → 2 colonnes (text, label) | Simplification ML |
| Encodage | target {0,4} → label {0,1} | Format MLlib |
| Nettoyage | URLs, @, #, ponctuation supprimés | Bruit éliminé |
| Sauvegarde | Parquet Snappy dans Docker → visible Windows | ~50-80 MB |

### ✅ Résumé des changements Docker vs Windows local

| Élément | Ancienne version | Nouvelle version Docker |
|---------|-----------------|------------------------|
| `import findspark` | ✅ présent | ❌ **supprimé** |
| `findspark.init()` | ✅ présent | ❌ **supprimé** |
| `.master(...)` | `'local[*]'` hardcodé | `SPARK_MASTER` depuis config |
| `.config('spark.driver.memory',...)` | `'4g'` hardcodé | `SPARK_DRIVER_MEMORY` depuis config |
| Chemins fichiers | Windows `Path().parent` | `/workspace` si Docker |
| `matplotlib.use(...)` | absent (mode interactif) | `'Agg'` (pas d'écran dans Docker) |
| Écriture Parquet | ❌ erreur winutils | ✅ fonctionne (Linux Docker) |

**Points importants pour la soutenance :**
- Le TF-IDF n'est pas sauvegardé séparément — il sera **intégré dans le pipeline ML complet** (Notebook 03)
- Le Parquet nettoyé sera rechargé par le Notebook 03 sans recalcul
- Le preprocessing est **identique** pour Sentiment140 et Apple Tweets (même UDF clean_text)
- La colonne Apple `tweet_text` sera simplement renommée `text` — aucun autre changement